# Unit 2.3: Image Enhancement

This notebook covers image enhancement techniques:
- Contrast enhancement (Histogram equalization, CLAHE)
- Brightness/Gamma correction
- Sharpening techniques (Unsharp mask)
- Noise reduction
- Image restoration

## Import Libraries

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Analyze Image

In [ ]:
# Load image
img_bgr = cv2.imread('sample.jpeg')
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

print(f"Image Shape: {img_gray.shape}")
print(f"Min Intensity: {img_gray.min()}")
print(f"Max Intensity: {img_gray.max()}")
print(f"Mean Intensity: {img_gray.mean():.2f}")
print(f"Std Deviation: {img_gray.std():.2f}")

# Calculate and display histogram
histogram = cv2.calcHist([img_gray], [0], None, [256], [0, 256])

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.imshow(img_gray, cmap='gray')
plt.title('Original Grayscale Image')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.plot(histogram, color='black', label='Histogram')
plt.xlabel('Pixel Intensity')
plt.ylabel('Frequency')
plt.title('Original Histogram')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

## 2. Histogram Equalization

In [ ]:
# Apply histogram equalization
equalized = cv2.equalizeHist(img_gray)

# Calculate histogram of equalized image
equalized_histogram = cv2.calcHist([equalized], [0], None, [256], [0, 256])

print(f"\nEqualized Image Statistics:")
print(f"Min Intensity: {equalized.min()}")
print(f"Max Intensity: {equalized.max()}")
print(f"Mean Intensity: {equalized.mean():.2f}")
print(f"Std Deviation: {equalized.std():.2f}")

# Visualization
plt.figure(figsize=(15, 8))

plt.subplot(2, 3, 1)
plt.imshow(img_gray, cmap='gray')
plt.title('Original Image')
plt.axis('off')

plt.subplot(2, 3, 2)
plt.imshow(equalized, cmap='gray')
plt.title('Histogram Equalized')
plt.axis('off')

# Show difference
difference = cv2.absdiff(img_gray.astype(np.float32), equalized.astype(np.float32))
plt.subplot(2, 3, 3)
plt.imshow(difference, cmap='hot')
plt.title('Difference Map')
plt.colorbar()

# Histograms
plt.subplot(2, 3, 4)
plt.plot(histogram, color='black')
plt.xlabel('Pixel Intensity')
plt.ylabel('Frequency')
plt.title('Original Histogram')
plt.grid(True, alpha=0.3)

plt.subplot(2, 3, 5)
plt.plot(equalized_histogram, color='blue')
plt.xlabel('Pixel Intensity')
plt.ylabel('Frequency')
plt.title('Equalized Histogram')
plt.grid(True, alpha=0.3)

# Cumulative Distribution Function (CDF)
cumsum_original = np.cumsum(histogram.flatten())
cdf_original = cumsum_original / cumsum_original[-1] * 255

cumsum_equalized = np.cumsum(equalized_histogram.flatten())
cdf_equalized = cumsum_equalized / cumsum_equalized[-1] * 255

plt.subplot(2, 3, 6)
plt.plot(cdf_original, label='Original CDF', color='black')
plt.plot(cdf_equalized, label='Equalized CDF', color='blue')
plt.xlabel('Pixel Intensity')
plt.ylabel('Cumulative Frequency')
plt.title('Cumulative Distribution Function')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Contrast Limited Adaptive Histogram Equalization (CLAHE)

In [ ]:
# Create CLAHE object
clahe_variants = []
clip_limits = [2, 5, 10]
tile_size = (8, 8)

plt.figure(figsize=(15, 4))

plt.subplot(1, 5, 1)
plt.imshow(img_gray, cmap='gray')
plt.title('Original')
plt.axis('off')

for idx, clip_limit in enumerate(clip_limits):
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
    clahe_result = clahe.apply(img_gray)
    clahe_variants.append(clahe_result)
    
    plt.subplot(1, 5, idx + 2)
    plt.imshow(clahe_result, cmap='gray')
    plt.title(f'CLAHE (clip={clip_limit})')
    plt.axis('off')

plt.tight_layout()
plt.suptitle('CLAHE with Different Clip Limits', fontsize=14, y=1.02)
plt.show()

print("\nCLAHE Statistics:")
for i, clip_limit in enumerate(clip_limits):
    img = clahe_variants[i]
    print(f"Clip Limit {clip_limit}: Mean={img.mean():.2f}, Std={img.std():.2f}, Min={img.min()}, Max={img.max()}")

## 4. Gamma Correction

In [ ]:
def apply_gamma_correction(image, gamma):
        """
        Apply gamma correction
        gamma < 1: brightens image
        gamma > 1: darkens image
        """
        inv_gamma = 1.0 / gamma
        table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in np.arange(0, 256)]).astype(np.uint8)
        return cv2.LUT(image, table)

# Apply different gamma values
gammas = [0.5, 1.0, 1.5, 2.0]
gamma_results = []

plt.figure(figsize=(15, 6))

for idx, gamma in enumerate(gammas):
    result = apply_gamma_correction(img_gray, gamma)
    gamma_results.append(result)
    
    plt.subplot(2, 4, idx + 1)
    plt.imshow(result, cmap='gray')
    plt.title(f'γ = {gamma}')
    if gamma < 1:
        plt.text(img_gray.shape[1]//2, -20, 'Brightening', ha='center', color='green')
    elif gamma > 1:
        plt.text(img_gray.shape[1]//2, -20, 'Darkening', ha='center', color='red')
    else:
        plt.text(img_gray.shape[1]//2, -20, 'Original', ha='center', color='black')
    plt.axis('off')

# Show gamma curves
for idx, gamma in enumerate(gammas):
    plt.subplot(2, 4, idx + 5)
    x = np.linspace(0, 1, 256)
    y = np.power(x, 1/gamma)
    plt.plot(y * 255)
    plt.xlabel('Input Intensity')
    plt.ylabel('Output Intensity')
    plt.title(f'Gamma Curve (γ={gamma})')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Gamma Correction Applied:")
for gamma in gammas:
    print(f"γ = {gamma}: {'Brightening' if gamma < 1 else 'Darkening' if gamma > 1 else 'Original'}")

## 5. Sharpening - Unsharp Mask

In [ ]:
# Unsharp Mask implementation
def unsharp_mask(image, sigma, alpha):
        """
        Apply unsharp mask sharpening
        Output = Original + α * (Original - Blurred)
        """
        blurred = cv2.GaussianBlur(image, (5, 5), sigma)
        sharpened = cv2.addWeighted(image.astype(np.float32), 1 + alpha, 
                                   blurred.astype(np.float32), -alpha, 0)
        return np.clip(sharpened, 0, 255).astype(np.uint8)

# Apply unsharp mask with different alpha values
alphas = [0.5, 1.0, 1.5, 2.0]
sigma = 1.0
sharpened_results = []

plt.figure(figsize=(15, 4))

plt.subplot(1, 5, 1)
plt.imshow(img_gray, cmap='gray')
plt.title('Original')
plt.axis('off')

for idx, alpha in enumerate(alphas):
    result = unsharp_mask(img_gray, sigma, alpha)
    sharpened_results.append(result)
    
    plt.subplot(1, 5, idx + 2)
    plt.imshow(result, cmap='gray')
    plt.title(f'α = {alpha}')
    plt.axis('off')

plt.tight_layout()
plt.suptitle('Unsharp Mask with Different Strength (σ=1.0)', fontsize=14, y=1.02)
plt.show()

## 6. Noise Reduction

In [ ]:
# Create noisy images
def add_gaussian_noise(image, mean=0, std=25):
        noise = np.random.normal(mean, std, image.shape)
        noisy = image.astype(np.float32) + noise
        return np.clip(noisy, 0, 255).astype(np.uint8)

noisy_img = add_gaussian_noise(img_gray, std=15)

# Apply different denoising methods
# 1. Gaussian Blur
gaussian_denoised = cv2.GaussianBlur(noisy_img, (5, 5), 1.0)

# 2. Bilateral Filter (edge-preserving)
bilateral_denoised = cv2.bilateralFilter(noisy_img, 5, 50, 50)

# 3. Non-Local Means Denoising
denoised_nlm = cv2.fastNlMeansDenoising(noisy_img, h=10, templateWindowSize=7, searchWindowSize=21)

# 4. Morphological operations
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
morpho_denoised = cv2.morphologyEx(noisy_img, cv2.MORPH_OPEN, kernel)

plt.figure(figsize=(15, 4))

plt.subplot(1, 5, 1)
plt.imshow(img_gray, cmap='gray')
plt.title('Original')
plt.axis('off')

plt.subplot(1, 5, 2)
plt.imshow(noisy_img, cmap='gray')
plt.title('Noisy Image')
plt.axis('off')

plt.subplot(1, 5, 3)
plt.imshow(gaussian_denoised, cmap='gray')
plt.title('Gaussian Blur')
plt.axis('off')

plt.subplot(1, 5, 4)
plt.imshow(bilateral_denoised, cmap='gray')
plt.title('Bilateral Filter')
plt.axis('off')

plt.subplot(1, 5, 5)
plt.imshow(denoised_nlm, cmap='gray')
plt.title('NLM Denoising')
plt.axis('off')

plt.tight_layout()
plt.show()

# Calculate PSNR (Peak Signal-to-Noise Ratio) for quality assessment
def calculate_psnr(original, noisy):
        mse = np.mean((original.astype(np.float32) - noisy.astype(np.float32)) ** 2)
        if mse == 0:
            return float('inf')
        max_pixel = 255.0
        psnr = 20 * np.log10(max_pixel / np.sqrt(mse))
        return psnr

print("\nDenoising Quality (PSNR in dB):")
print(f"Noisy Image: {calculate_psnr(img_gray, noisy_img):.2f} dB")
print(f"Gaussian Blur: {calculate_psnr(img_gray, gaussian_denoised):.2f} dB")
print(f"Bilateral Filter: {calculate_psnr(img_gray, bilateral_denoised):.2f} dB")
print(f"NLM Denoising: {calculate_psnr(img_gray, denoised_nlm):.2f} dB")
print(f"Morphological: {calculate_psnr(img_gray, morpho_denoised):.2f} dB")

## 7. Combined Enhancement Pipeline

In [ ]:
# Create a degraded image (low contrast, noisy)
degraded = cv2.cvtColor(cv2.imread('sample.jpeg'), cv2.COLOR_BGR2GRAY)

# Make it low contrast
low_contrast = cv2.convertScaleAbs(degraded, alpha=0.5, beta=50)
low_contrast_noisy = add_gaussian_noise(low_contrast, std=20)

# Apply enhancement pipeline
step1 = cv2.GaussianBlur(low_contrast_noisy, (5, 5), 1.0)  # Denoise
step2 = cv2.equalizeHist(step1)  # Enhance contrast
step3 = unsharp_mask(step2, 1.0, 0.8)  # Sharpen

plt.figure(figsize=(15, 8))

plt.subplot(2, 3, 1)
plt.imshow(degraded, cmap='gray')
plt.title('Original')
plt.axis('off')

plt.subplot(2, 3, 2)
plt.imshow(low_contrast, cmap='gray')
plt.title('Low Contrast')
plt.axis('off')

plt.subplot(2, 3, 3)
plt.imshow(low_contrast_noisy, cmap='gray')
plt.title('Low Contrast + Noisy')
plt.axis('off')

plt.subplot(2, 3, 4)
plt.imshow(step1, cmap='gray')
plt.title('Step 1: Denoising')
plt.axis('off')

plt.subplot(2, 3, 5)
plt.imshow(step2, cmap='gray')
plt.title('Step 2: Histogram Equalization')
plt.axis('off')

plt.subplot(2, 3, 6)
plt.imshow(step3, cmap='gray')
plt.title('Step 3: Sharpening (Final)')
plt.axis('off')

plt.tight_layout()
plt.suptitle('Image Enhancement Pipeline', fontsize=14, y=1.00)
plt.show()

print("\nEnhancement Pipeline Statistics:")
print(f"Original Mean: {degraded.mean():.2f}")
print(f"Degraded Mean: {low_contrast_noisy.mean():.2f}")
print(f"Enhanced Mean: {step3.mean():.2f}")
print(f"\nOriginal Std: {degraded.std():.2f}")
print(f"Degraded Std: {low_contrast_noisy.std():.2f}")
print(f"Enhanced Std: {step3.std():.2f}")

## 8. Advanced Techniques

In [ ]:
# Morphological Operations
kernel_sizes = [3, 5, 7]
kernels = [cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)) for k in kernel_sizes]

noisy_binary = cv2.threshold(noisy_img, 128, 255, cv2.THRESH_BINARY)[1]

plt.figure(figsize=(15, 4))

plt.subplot(1, 4, 1)
plt.imshow(noisy_binary, cmap='gray')
plt.title('Binary Image')
plt.axis('off')

for idx, (k_size, kernel) in enumerate(zip(kernel_sizes, kernels)):
    # Opening: erosion followed by dilation
    opening = cv2.morphologyEx(noisy_binary, cv2.MORPH_OPEN, kernel)
    
    plt.subplot(1, 4, idx + 2)
    plt.imshow(opening, cmap='gray')
    plt.title(f'Opening ({k_size}x{k_size})')
    plt.axis('off')

plt.tight_layout()
plt.show()

# Image comparison
print("\nMorphological Operations:")
print("- Opening: Erosion → Dilation (removes small objects)")
print("- Closing: Dilation → Erosion (fills small holes)")
print("- Gradient: Dilation - Erosion (extracts edges)")
print("- TopHat: Original - Opening (extracts small elements)")

## Summary of Enhancement Techniques

### Contrast Enhancement:
- **Histogram Equalization**: Simple, can over-amplify noise
- **CLAHE**: Better control, preserves local details

### Brightness Correction:
- **Gamma Correction**: Mimics camera response
- Useful for fixing under/over-exposed images

### Sharpening:
- **Unsharp Mask**: Effective and controllable
- Good for fine detail enhancement

### Denoising:
- **Gaussian Blur**: Simple but blurs edges
- **Bilateral Filter**: Preserves edges
- **NLM**: Best quality, computationally expensive

### Best Practices:
1. Always start with denoising
2. Apply contrast enhancement
3. Add sharpening if needed
4. Adjust gamma for final appearance
5. Validate with visual inspection